In [1]:
#!/usr/bin/env python3
import sys
import subprocess

def print_system_info():
    print("=== System Information ===")
    print(f"Platform: {sys.platform}")
    print(f"Python version: {sys.version.split()[0]}")
    print()

def print_torch_info():
    try:
        import torch
        print("=== PyTorch Information ===")
        print(f"  PyTorch version: {torch.__version__}")
        cuda_ok = torch.cuda.is_available()
        print(f"  CUDA available: {cuda_ok}")
        if cuda_ok:
            print(f"  CUDA version (from torch): {torch.version.cuda}")
            print(f"  cuDNN version: {torch.backends.cudnn.version()}")
            n_gpus = torch.cuda.device_count()
            print(f"  Number of GPUs: {n_gpus}")
            for i in range(n_gpus):
                name = torch.cuda.get_device_name(i)
                mem_alloc = torch.cuda.memory_allocated(i) / (1024**3)
                mem_reserved = torch.cuda.memory_reserved(i) / (1024**3)
                print(f"    GPU {i}: {name}")
                print(f"      Allocated: {mem_alloc:.2f} GB, Reserved: {mem_reserved:.2f} GB")
        print()
    except ImportError:
        print("PyTorch is not installed.\n")

def print_tf_info():
    try:
        import tensorflow as tf
        print("=== TensorFlow Information ===")
        print(f"  TensorFlow version: {tf.__version__}")
        gpus = tf.config.list_physical_devices('GPU')
        print(f"  GPUs detected by TensorFlow: {len(gpus)}")
        for gpu in gpus:
            details = tf.config.experimental.get_device_details(gpu)
            print(f"    {details.get('device_name', gpu)}")
        print()
    except ImportError:
        print("TensorFlow is not installed.\n")

def print_nvidia_smi():
    print("=== NVIDIA-SMI Output ===")
    try:
        res = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        print(res.stdout.strip())
    except Exception as e:
        print(f"  nvidia-smi not found or failed: {e}")

if __name__ == "__main__":
    print_system_info()
    print_torch_info()
    print_tf_info()
    print_nvidia_smi()


=== System Information ===
Platform: win32
Python version: 3.11.9

=== PyTorch Information ===
  PyTorch version: 2.5.1+cu118
  CUDA available: True
  CUDA version (from torch): 11.8
  cuDNN version: 90100
  Number of GPUs: 1
    GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU
      Allocated: 0.00 GB, Reserved: 0.00 GB

=== TensorFlow Information ===
  TensorFlow version: 2.19.0
  GPUs detected by TensorFlow: 0

=== NVIDIA-SMI Output ===
Tue Apr 22 13:11:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.36                 Driver Version: 566.36         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |   

In [4]:
#!/usr/bin/env python3
import sys
import platform
import subprocess

def check_python():
    print(f"Python: {sys.version.split()[0]} ({platform.python_implementation()})")

def check_torch():
    try:
        import torch
        print(f"PyTorch: {torch.__version__}")
        print(f"  - CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"  - CUDA version: {torch.version.cuda}")
            print(f"  - cuDNN version: {torch.backends.cudnn.version()}")
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                total_mem_gb = props.total_memory / (1024**3)
                print(f"  - GPU #{i}: {props.name} ({total_mem_gb:.1f} GB)")
    except ImportError:
        print("PyTorch: not installed")

def check_tensorflow():
    try:
        import tensorflow as tf
        print(f"TensorFlow: {tf.__version__}")
        gpus = tf.config.list_physical_devices('GPU')
        print(f"  - GPUs detected by TF: {[gpu.name for gpu in gpus]}")
    except ImportError:
        print("TensorFlow: not installed")

def check_nvidia_smi():
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            encoding='utf-8'
        ).strip().splitlines()
        print("nvidia‑smi:")
        for line in output:
            idx, name, drv, mem, util = [x.strip() for x in line.split(",")]
            print(f"  - GPU #{idx}: {name}, driver {drv}, {mem} MB total, {util}% util")
    except FileNotFoundError:
        print("nvidia‑smi: not found (NVIDIA driver may not be installed)")
    except subprocess.CalledProcessError as e:
        print(f"nvidia‑smi error: {e}")

def main():
    print("="*60)
    check_python()
    print("-"*60)
    check_torch()
    print("-"*60)
    check_tensorflow()
    print("-"*60)
    check_nvidia_smi()
    print("="*60)

if __name__ == "__main__":
    main()


Python: 3.11.9 (CPython)
------------------------------------------------------------
PyTorch: 2.5.1+cu118
  - CUDA available: True
  - CUDA version: 11.8
  - cuDNN version: 90100
  - GPU #0: NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB)
------------------------------------------------------------
TensorFlow: 2.19.0
  - GPUs detected by TF: []
------------------------------------------------------------
nvidia‑smi:
  - GPU #0: NVIDIA GeForce RTX 4060 Laptop GPU, driver 566.36, 8188 MB total, 13% util
